In [2]:
import os
import pandas as pd

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='rdkit')



smi_directory = '/mnt/ligandpro/db/PDB/pdb2/lig/smi_files'

data = []

for filename in os.listdir(smi_directory):
    if filename.endswith(".smi"):
        filepath = os.path.join(smi_directory, filename)
        pdb_id = filename.replace("pdb", "").replace(".smi", "")
        with open(filepath, 'r') as file:
            for line in file:
                parts = line.strip().split()
                if len(parts) == 3:
                    smiles, _, ligand_chain = parts
                    het, chain = ligand_chain.split('_')
                    data.append({'SMILES': smiles, 'PDB': pdb_id, 'HET': het, 'Chain': chain})

df = pd.DataFrame(data)



In [3]:
df.shape

(314789, 4)

In [4]:
import json

with open('../data/trash_ligands.json', 'r') as f:
    trash_ligands = json.load(f)

initial_count = len(df)

df_filtered = df[~df['HET'].isin(trash_ligands)]

final_count = len(df_filtered)

removed_count = initial_count - final_count

print(f"Total records before filtering: {initial_count}")
print(f"Total records after filtering: {final_count}")
print(f"Total records removed: {removed_count}")




Total records before filtering: 314789
Total records after filtering: 314469
Total records removed: 320


In [5]:
df

,SMILES,PDB,HET,Chain
0,[Fe@@]123n4c5=CC6=[N]3C(=CC3=[C](C(=C(C=C7[N]1...,1dy7,DHE,B
1,C=O,1dy7,CMO,A
2,C=O,1dy7,CMO,B
3,[Fe@@]123[N]4=C5C=c6n3c(=CC3=[N]2C(=CC2=[C](C(...,1dy7,DHE,A
4,c1(N[C]2=NC=C(C(F)(F)F)C(=[N]=2)N[C@@H]2CCC[C@...,6yvy,P1E,C
...,...,...,...,...
314784,C(C(=O)O)C(=O)O,5q53,MLI,A
314785,C(=O)(O)c1ccc(cc1)O,7dqb,PHB,A
314786,C(=O)(O)c1ccc(cc1)O,7dqb,PHB,B
314787,N1C(=O)[C@@H](C2=C3[C@@H](C=CC(=C3)F)N=C2)CC1=...,6pz1,AOJ,B


In [7]:
import pandas as pd
import datamol as dm
from joblib import Parallel, delayed
from tqdm import tqdm

smiles_column = "SMILES"

def _preprocess(i, row):
    try:
        dm.disable_rdkit_log()

        mol = dm.to_mol(row[smiles_column], ordered=True)
        if mol is None:
            return None

        mol = dm.fix_mol(mol)
        mol = dm.sanitize_mol(mol, sanifix=True, charge_neutral=False)
        mol = dm.standardize_mol(mol, disconnect_metals=False, normalize=True, reionize=True, uncharge=True, stereo=True)

        row["standard_smiles"] = dm.standardize_smiles(dm.to_smiles(mol))
        return row

    except Exception as e:
        #print(f"Error processing molecule {i}: {str(e)}")
        return None

results = Parallel(n_jobs=-1)(delayed(_preprocess)(i, row) for i, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)))

df_standardized = pd.DataFrame([res for res in results if res is not None])

successful_conversions = len(df_standardized)
total_molecules = len(df_filtered)
conversion_rate = (successful_conversions / total_molecules) * 100


print(f"Total molecules: {total_molecules}")
print(f"Successfully standardized molecules: {successful_conversions}")
print(f"Conversion success rate: {conversion_rate:.2f}%")



  1%|          | 2240/314469 [00:01<03:19, 1562.97it/s]

Error processing molecule 1976: Explicit valence for atom # 13 O, 3, is greater than permitted


 13%|█▎        | 39872/314469 [00:16<01:03, 4332.31it/s]

Error processing molecule 38557: Explicit valence for atom # 11 O, 3, is greater than permitted
Error processing molecule 38558: Explicit valence for atom # 11 O, 3, is greater than permitted
Error processing molecule 39084: Explicit valence for atom # 14 O, 3, is greater than permitted


 35%|███▍      | 109088/314469 [01:26<05:09, 663.35it/s]

...

Error processing molecule 184262: Explicit valence for atom # 20 O, 3, is greater than permitted


 59%|█████▉    | 185248/314469 [03:13<02:41, 800.25it/s]

Error processing molecule 185178: Explicit valence for atom # 13 O, 3, is greater than permitted
Error processing molecule 185179: Explicit valence for atom # 13 O, 3, is greater than permitted


100%|██████████| 314469/314469 [05:49<00:00, 900.92it/s] 


Total molecules: 314469
Successfully standardized molecules: 293621
Conversion success rate: 93.37%


In [8]:
df_standardized

,SMILES,PDB,HET,Chain,standard_smiles
1,C=O,1dy7,CMO,A,C=O
2,C=O,1dy7,CMO,B,C=O
9,[P@]1(=O)(O)O[C@@H]2[C@@H](COP(=O)=O)O[C@H]([C...,6ufh,A23,B,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P+](=O)[O-])[C@...
10,[P@@]1(=O)(O)O[C@@H]2[C@@H](COP(=O)=O)O[C@H]([...,6ufh,CCC,A,Nc1ccn([C@@H]2O[C@H](CO[P+](=O)[O-])[C@H]3O[P@...
11,c12c(c(ncn1)N)ncn2[C@@H]1O[C@H](CO[P@](=O)(O[P...,5ojn,9X8,A,C[C@H](S)O[C@@H]1[C@H](O)[C@@H](CO[P@@](=O)(O)...
...,...,...,...,...,...
314784,C(C(=O)O)C(=O)O,5q53,MLI,A,O=C(O)CC(=O)O
314785,C(=O)(O)c1ccc(cc1)O,7dqb,PHB,A,O=C(O)c1ccc(O)cc1
314786,C(=O)(O)c1ccc(cc1)O,7dqb,PHB,B,O=C(O)c1ccc(O)cc1
314787,N1C(=O)[C@@H](C2=C3[C@@H](C=CC(=C3)F)N=C2)CC1=...,6pz1,AOJ,B,O=C1C[C@H](C2=C3C=C(F)C=C[C@@H]3N=C2)C(=O)N1.O...
